RunnableParallel

In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [4]:
chat_template_books = ChatPromptTemplate.from_template(
    """
    Suggest three of the best intermediate-level {programming_language} books.
    Answer only by listing the books.
    """
)

chat_template_projects = ChatPromptTemplate.from_template(
    """
    Suggest three interesting {programming_language} projects suitable for intermediate-level programmers.
    Answer only by listing the projects.
    """
)

In [5]:
chat = ChatGroq(model_name="llama-3.1-8b-instant",
                model_kwargs= {'seed':365},
                temperature = 0,
                max_tokens= 100)

In [6]:
string_parser = StrOutputParser()

In [7]:
chain_books = chat_template_books | chat | string_parser

chain_projects = chat_template_projects | chat | string_parser

In [8]:
chain_parallel = RunnableParallel({'books':chain_books, 'projects':chain_projects})

In [9]:
chain_parallel.invoke({'programming_language':'Python'})

{'books': '1. "Automate the Boring Stuff with Python" by Al Sweigart\n2. "Python Crash Course" by Eric Matthes\n3. "Learning Python" by Mark Lutz',
 'projects': '1. Web Scraper with Database Integration\n2. Chatbot using Natural Language Processing (NLP)\n3. Game Development with Pygame or Pyglet'}

In [11]:
chain_parallel.get_graph().print_ascii()

            +-------------------------------+            
            | Parallel<books,projects>Input |            
            +-------------------------------+            
                   **               **                   
                ***                   ***                
              **                         **              
+--------------------+            +--------------------+ 
| ChatPromptTemplate |            | ChatPromptTemplate | 
+--------------------+            +--------------------+ 
           *                                 *           
           *                                 *           
           *                                 *           
     +----------+                      +----------+      
     | ChatGroq |                      | ChatGroq |      
     +----------+                      +----------+      
           *                                 *           
           *                                 *           
           *  

In [15]:
%%time
chain_books.invoke({'programming_language':'Python'})

CPU times: total: 31.2 ms
Wall time: 1.53 s


'1. "Automate the Boring Stuff with Python" by Al Sweigart\n2. "Python Crash Course" by Eric Matthes\n3. "Learning Python" by Mark Lutz'

In [16]:
%%time
chain_projects.invoke({'programming_language':'Python'})

CPU times: total: 0 ns
Wall time: 667 ms


'1. Web Scraper with Database Integration\n2. Chatbot using Natural Language Processing (NLP)\n3. Game Development with Pygame or Pyglet'

In [17]:
%%time
chain_parallel.invoke({'programming_language':'Python'})

CPU times: total: 15.6 ms
Wall time: 3.8 s


{'books': '1. "Automate the Boring Stuff with Python" by Al Sweigart\n2. "Python Crash Course" by Eric Matthes\n3. "Learning Python" by Mark Lutz',
 'projects': '1. Web Scraper with Database Integration\n2. Chatbot using Natural Language Processing (NLP)\n3. Game Development with Pygame or Pyglet'}